# Store Sales Time Series Forecasting TC1 Human Baseline

This notebook defines the TC1 from-scratch human baseline for the **store-sales-time-series-forecasting** competition.
Each preprocessing section is tagged with its `CA-XXXXXX` action ID from `candidate_actions.json`.

## Load Data

In [ ]:
import kagglehub
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path(kagglehub.competition_download('store-sales-time-series-forecasting'))

required_files = [
    'train.csv', 'test.csv', 'stores.csv', 'oil.csv',
    'holidays_events.csv', 'sample_submission.csv',
]
missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files under {DATA_DIR}: {missing}')

train            = pd.read_csv(DATA_DIR / 'train.csv')
test             = pd.read_csv(DATA_DIR / 'test.csv')
stores           = pd.read_csv(DATA_DIR / 'stores.csv')
oil              = pd.read_csv(DATA_DIR / 'oil.csv')
holidays_events  = pd.read_csv(DATA_DIR / 'holidays_events.csv')
sample_sub       = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train shape:', train.shape)
print('test shape: ', test.shape)
print('stores:', stores.shape, '  oil:', oil.shape, '  holidays:', holidays_events.shape)
train.head(2)

## CA-000002 — Parse Date Column

The `date` column is stored as a string in all tables.
Parsing it to `datetime64` is a universal first step before any calendar
decomposition, join, or lag feature.

Applied to `train`, `test`, `oil`, and `holidays_events`.

In [ ]:
# CA-000002: PARSE_DATETIME (date column, all tables)
train['date']           = pd.to_datetime(train['date'])
test['date']            = pd.to_datetime(test['date'])
oil['date']             = pd.to_datetime(oil['date'])
holidays_events['date'] = pd.to_datetime(holidays_events['date'])

print('train date range:', train['date'].min(), 'to', train['date'].max())
print('test  date range:', test['date'].min(),  'to', test['date'].max())
train[['date', 'store_nbr', 'family', 'sales']].head(3)

## CA-000004 — Join Stores Metadata

Joining the `stores.csv` lookup table adds city, state, store type, and cluster
— categorical store-level features unavailable in `train.csv` but used by
virtually all strong baselines.

In [ ]:
# CA-000004: JOIN_LOOKUP (stores, on=store_nbr, how=left)
train = train.merge(stores, on='store_nbr', how='left')
test  = test.merge(stores, on='store_nbr', how='left')

print('Columns added from stores:', [c for c in stores.columns if c != 'store_nbr'])
print('train shape:', train.shape)
train[['store_nbr', 'city', 'state', 'type', 'cluster']].head(3)

## CA-000007 — Join Oil Prices

Oil price is a macro-economic exogenous feature strongly correlated with
consumer spending in Ecuador. Joined by date using a left join to preserve
all training rows (oil has no entries for weekends/holidays).

In [ ]:
# CA-000007: JOIN_LOOKUP (oil, on=date, how=left)
train = train.merge(oil, on='date', how='left')
test  = test.merge(oil, on='date', how='left')

print('NaN oil prices in train:', train['dcoilwtico'].isna().sum())
print('train shape:', train.shape)

## CA-000011 — Join Holiday Events

Ecuador holiday and event data is a first-class exogenous signal.
Holidays cause measurable sales spikes and dips and are referenced in the
official competition description.

Left join preserves all non-holiday rows.

In [ ]:
# CA-000011: JOIN_LOOKUP (holidays_events, on=date, how=left)
# Rename 'type' -> 'holiday_type' to avoid collision with stores 'type' column.
# Keep only one holiday row per date (national > regional > local priority) to
# prevent a many-to-many date join that would fan out train rows.
holiday_cols = ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']
locale_order = {'National': 0, 'Regional': 1, 'Local': 2}
holidays_renamed = (
    holidays_events[holiday_cols]
    .rename(columns={'type': 'holiday_type'})
    .assign(_locale_rank=lambda df: df['locale'].map(locale_order).fillna(3))
    .sort_values('_locale_rank')
    .drop_duplicates(subset='date', keep='first')
    .drop(columns='_locale_rank')
)

train = train.merge(holidays_renamed, on='date', how='left')
test  = test.merge(holidays_renamed, on='date', how='left')

print('Columns added from holidays:', [c for c in holidays_renamed.columns if c != 'date'])
print('train shape:', train.shape)
print('Holiday rows in train:', train['holiday_type'].notna().sum())

## CA-000018 — Oil Price Interpolation

Oil prices have systematic gaps on weekends and holidays after the date-keyed
left join. Linear interpolation fills these NaN gaps to produce a complete
daily series; `bfill` handles any leading NaN values.

In [ ]:
# CA-000018: APPLY_EXPRESSION (linear interpolation of dcoilwtico)
train['dcoilwtico'] = train['dcoilwtico'].interpolate(method='linear').bfill()
test['dcoilwtico']  = test['dcoilwtico'].interpolate(method='linear').bfill()

print('NaN oil prices after interpolation — train:', train['dcoilwtico'].isna().sum())
print('NaN oil prices after interpolation — test: ', test['dcoilwtico'].isna().sum())

## CA-000024 — Filter Transferred Holidays

Transferred holidays are moved to another date; keeping them at their original
date introduces phantom holiday signals on non-event days.

After the holiday left join, ordinary non-holiday rows have missing
`transferred` values. The cleanup must therefore keep missing-or-False rows
and only remove the transferred holiday label from moved-holiday dates.

In [ ]:
# CA-000024: FILTER_ROWS (transferred.isna() | (transferred == False))
# I keep normal days and clear only the moved-holiday labels.
hol_attr_cols = ['holiday_type', 'locale', 'locale_name', 'description', 'transferred']

transferred_mask_train = train['transferred'] == True
transferred_mask_test  = test['transferred'] == True

train.loc[transferred_mask_train, hol_attr_cols] = np.nan
test.loc[transferred_mask_test,  hol_attr_cols] = np.nan

print(f'Cleared {transferred_mask_train.sum()} transferred-holiday rows in train')
print(f'Cleared {transferred_mask_test.sum()} transferred-holiday rows in test')

## CA-000031 — Calendar Date Features

Calendar decomposition into year, month, day-of-week, and quarter captures
seasonality and trend at multiple temporal granularities — the most common
feature engineering step in this competition.

*Alternative*: compact variant with only month and dayofweek (CA-000034).

In [ ]:
# CA-000031: DATE_PART_FEATURE (year, month, dayofweek, quarter; drop_original=False)
for df in [train, test]:
    df['year']      = df['date'].dt.year
    df['month']     = df['date'].dt.month
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter']   = df['date'].dt.quarter

print('Calendar features added: year, month, dayofweek, quarter')
train[['date', 'year', 'month', 'dayofweek', 'quarter']].head(3)


## CA-000044 — Encode Family Column

The `family` column (33 product categories) is a high-cardinality categorical
that tree-based models require as an integer. Label encoding is the dominant
approach in gradient-boosted tree notebooks.

*Alternative*: one-hot encoding via `pd.get_dummies` (CA-000048).

In [ ]:
# CA-000044: ENCODE_CATEGORICAL (label, family, fit_scope=full_data)
le = LabelEncoder()
le.fit(list(train['family'].values) + list(test['family'].values))
train['family'] = le.transform(train['family'].values)
test['family']  = le.transform(test['family'].values)

print('Family encoded. Unique codes:', train['family'].nunique())
train[['family']].head(3)


## CA-000052 — Drop id Column

The `id` column is a submission row identifier with no predictive value.
It must be removed from the feature matrix before training.

In [ ]:
# CA-000052: DROP_COLUMNS (id)
test_ids = test['id'].copy()

train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

y_train = train['sales'].values
X_train = train.drop(columns=['sales', 'date'])
X_test  = test.drop(columns=['date'])

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)

## Final Shape Check

In [ ]:
print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('y_train shape:', y_train.shape)

assert X_train.shape[1] == X_test.shape[1], 'Column mismatch between train and test'
assert len(X_train) == len(y_train), 'Row mismatch between features and target'
assert len(X_test) == len(sample_sub), 'Test/submission mismatch'
print('All shape checks passed.')
X_train.head(2)
